# Text Classification

Let's start with the Crowdflower Airline Sentiment Twitter dataset. 

    A sentiment analysis job about the problems of each major U.S. airline. Twitter data was scraped from February of 2015 and contributors were asked to first classify positive, negative, and neutral tweets, followed by categorizing negative reasons (such as "late flight" or "rude service").
    
This dataset was produced by human categorization. But if you were working for an airline, you can easily imagine that you might want to assess customer sentiment (and reasons for negative sentiment) week by week without manually categorizing thousands of tweets. To do that, you might need to train some kind of predictive model.

In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_validate
from pathlib import Path
import math

### Loading the data

Let's start by loading the data and doing some exploratory analysis.

In [4]:
tweetpath = Path('./flight_sentiment.tsv')

tweets = pd.read_csv(tweetpath, sep ='\t')

print(tweets.shape)

tweets.head()

(14640, 5)


,airline_sentiment,negativereason,retweet_count,airline,text
0,neutral,NaN,0,Virgin America,@VirginAmerica What @dhepburn said.
1,positive,NaN,0,Virgin America,@VirginAmerica plus you've added commercials t...
2,neutral,NaN,0,Virgin America,@VirginAmerica I didn't today... Must mean I n...
3,negative,Bad Flight,0,Virgin America,@VirginAmerica it's really aggressive to blast...
4,negative,Can't Tell,0,Virgin America,@VirginAmerica and it's a really big bad thing...


#### Exploratory data analysis

It's always a good idea to inspect the data before modeling it.

1. Find out what the top three reasons for unhappiness are. 

2. Visualize the distribution of positive, neutral, and negative ```airline_sentiment``` as a bar graph.

3. Examine the full texts of 5 tweets in the "Bad Flight" category, to see what that actually means.

To make full texts a little easier to inspect I'm going to start by telling Pandas to display the full column width.

In [6]:
pd.set_option('display.max_colwidth', None)

### Now create a term-document matrix (wordcounts)

In [8]:
vectorizer = CountVectorizer(max_features = 4000)

I'm going to start occasionally using an alternative way to refer to Pandas columns, which is simply

    ```df.columnname```
    instead of
    ```df['columnname']

This can occasionally be confusing. If you name a column something like "shape" for instance, you could be in trouble! But it's a lot quicker to type.

In [10]:
sparse_wordcounts = vectorizer.fit_transform(tweets.text)
wordcounts = sparse_wordcounts.toarray()
tweetwords = pd.DataFrame(wordcounts, columns = vectorizer.get_feature_names_out())
tweetwords.head()

,00,000,0016,00pm,02,03,05,05am,05pm,08,...,yr,yrs,yuma,yup,yvr,yyz,zero,zkatcher,zone,zurich
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


#### Discussion

If we really cared about maximizing accuracy it would probably be worthwhile to think hard about the strategy we're using to count words. For instance, how will the CountVectorizer handle a hashtag like "#soreback"? How might we ideally want to handle it?

Let's use the **Dunning’s log-likelihood statistic (G²)** to select some of the words most likely to predict positive or negative sentiment.

In [13]:
negative_words = tweetwords.loc[tweets.airline_sentiment == 'negative', :].sum(axis = 'rows')
positive_words = tweetwords.loc[tweets.airline_sentiment == 'positive', :].sum(axis = 'rows')

In [14]:
# The function "get_dunnings" computes Dunning’s log-likelihood statistic (G²) for a single word across two series
# It answers the question:
# “Is this word used disproportionately more in series1 or series2?”

# And it returns a signed value:
#  - Positive → word is over-represented in series1
#  - Negative → word is over-represented in series2
#  - Near zero → no meaningful difference

# This is commonly used in corpus linguistics, NLP, and keyword analysis.

def get_dunnings(word, series1, series2):
    # Build the observed contingency table, Row 1 = how often the word appears, Row 2 = how often everything else appears
    observed = pd.DataFrame({'series1': [series1[word], sum(series1) - series1[word]],
                          'series2': [series2[word], sum(series2) - series2[word]]},
                        index = ['word', 'all_others'])
    print(observed)
    print('\n')

    # Total number of tokens in both series
    total_words = observed.to_numpy().sum()

    # Row-wise total counts
    observed['word_totals'] = observed.sum(axis = 1)
    print(observed)
    print('\n')

    # Column-wise total counts
    observed = pd.concat([observed, pd.DataFrame([observed.sum(axis = 0).rename(index = 'group_totals')])])
    print(observed)
    print('\n')

    # Clean up the cell value of observed['word_totals', 'group_totals'] as 0, to avoid double-counting
    observed.iat[2,2] = 0
    print(observed)
    print('\n')

    # Convert totals into proportions, word_totals = probability of word vs all others, group_totals = probability of series1 vs series2
    observed['word_totals'] = observed['word_totals'] / sum(observed['word_totals'])
    observed.loc['group_totals', : ] = observed.loc['group_totals', : ] / sum(observed.loc['group_totals', : ])
    print(observed)
    print('\n')

    # Compute expected frequencies, expected = row probability × column probability × total words
    expected = np.outer(observed['word_totals'][0:2], observed.loc['group_totals', : ][0:2])
    print("Expected Values:\n")
    print(expected)
    print('\n')
    expected = pd.DataFrame(expected, index = ['word', 'all_others'], columns = ['series1', 'series2'])
    print("Total_words:\n")
    print(total_words)
    print('\n')
    expected = expected * total_words
    print(expected)
    print('\n')

    # Compute Dunning’s G² statistic: G² = 2 × \sigma (Observed × \log (Obversed / Expected))
    G = 0
    for i in range(2):
        for j in range(2):
            O = observed.iat[i, j] + .000001
            E = expected.iat[i, j] + .000001
            G = G + O * math.log(O / E)
    
    if (observed.iat[0, 0] / sum(observed.iloc[0: 2, 0])) < (observed.iat[0, 1] / sum(observed.iloc[0 : 2, 1])):
        G = -G    # we provide a signed version of the statistic to distinguish
                  # overrepresentation in the two categories
    
    return 2 * G


w = "excellent"
print("target word: ", w)
G = get_dunnings(w, positive_words, negative_words)

target word:  excellent
            series1  series2
word             24        4
all_others    29676   164066


            series1  series2  word_totals
word             24        4           28
all_others    29676   164066       193742


              series1  series2  word_totals
word               24        4           28
all_others      29676   164066       193742
group_totals    29700   164070       193770


              series1  series2  word_totals
word               24        4           28
all_others      29676   164066       193742
group_totals    29700   164070            0


                   series1        series2  word_totals
word             24.000000       4.000000     0.000145
all_others    29676.000000  164066.000000     0.999855
group_totals      0.153275       0.846725     0.000000


Expected Values:

[[2.21483512e-05 1.22352862e-04]
 [1.53252352e-01 8.46603146e-01]]


Total_words:

193770


                 series1        series2
word            4.291686      2

C:\Users\leoxi\AppData\Local\Temp\ipykernel_5884\428194829.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.1532745006967023' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  observed.loc['group_totals', : ] = observed.loc['group_totals', : ] / sum(observed.loc['group_totals', : ])
C:\Users\leoxi\AppData\Local\Temp\ipykernel_5884\428194829.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.8467254993032978' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  observed.loc['group_totals', : ] = observed.loc['group_totals', : ] / sum(observed.loc['group_totals', : ])


In [15]:
# Version of the above function without annotations

def get_dunnings(word, series1, series2):
    observed = pd.DataFrame({'series1': [series1[word], sum(series1) - series1[word]],
                          'series2': [series2[word], sum(series2) - series2[word]]},
                        index = ['word', 'all_others'])
    total_words = observed.to_numpy().sum()
    observed['word_totals'] = observed.sum(axis = 1)
    #observed = observed.append(observed.sum(axis = 0).rename(index = 'group_totals'))
    observed = pd.concat([observed, pd.DataFrame([observed.sum(axis = 0).rename(index = 'group_totals')])])
    observed.iat[2,2] = 0
    observed['word_totals'] = observed['word_totals'] / sum(observed['word_totals'])
    observed.loc['group_totals', : ] = observed.loc['group_totals', : ] / sum(observed.loc['group_totals', : ])
    expected = np.outer(observed['word_totals'][0:2], observed.loc['group_totals', : ][0:2])
    expected = pd.DataFrame(expected, index = ['word', 'all_others'], columns = ['series1', 'series2'])
    expected = expected * total_words
    
    G = 0
    for i in range(2):
        for j in range(2):
            O = observed.iat[i, j] + .000001
            E = expected.iat[i, j] + .000001
            G = G + O * math.log(O / E)
    
    if (observed.iat[0, 0] / sum(observed.iloc[0: 2, 0])) < (observed.iat[0, 1] / sum(observed.iloc[0 : 2, 1])):
        G = -G    # we provide a signed version of the statistic to distinguish
                  # overrepresentation in the two categories

    observed.loc['group_totals', :] = (
    observed.loc['group_totals', :] / sum(observed.loc['group_totals', :])
    ).astype(float)
    
    return 2 * G

In [77]:
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

dunningslist = []
for w in vectorizer.get_feature_names_out():
    G = get_dunnings(w, positive_words, negative_words)
    dunningslist.append(G)
dunnings = pd.Series(dunningslist, index=vectorizer.get_feature_names_out())
dunnings = dunnings.sort_values()

warnings.filterwarnings('default', category=FutureWarning)  # Re-enable after

In [17]:
dunnings[0:20]

no          -165.609085
hours       -165.077254
hold        -154.163694
not         -138.953626
cancelled   -130.925833
delayed      -86.071118
flightled    -83.028421
worst        -80.248818
hour         -74.138316
call         -73.718327
why          -65.215920
been         -59.384149
is           -58.140458
hrs          -56.984742
usairways    -55.275240
waiting      -53.808645
can          -48.977909
because      -47.704688
told         -47.414103
luggage      -47.414103
dtype: float64

No one mentions "luggage" or "hours" if they're happy!

In [19]:
dunnings[-20:]

excellent          68.405151
wonderful          68.405151
kudos              79.253573
rock               82.791590
appreciate        103.413990
good              110.310671
http              128.769954
co                145.210322
much              158.874557
virginamerica     171.634044
best              192.788829
amazing           218.934091
you               238.134708
awesome           280.567625
love              287.073472
southwestair      319.243684
jetblue           431.915012
great             553.288110
thanks           1218.129806
thank            1284.022918
dtype: float64

Let's select a list of 400 highly predictive features.

In [21]:
extremefeatures = list(dunnings.index[0:100].values) + list(dunnings.index[-100: ].values)

### A first model: Naive Bayes

To keep this simple to start with, let's create a dataframe with even numbers of positive and negative tweets. Let's say 200 positive ones and 200 negative ones.

In [24]:
positive_tweets = tweetwords.loc[tweets.airline_sentiment == 'positive', :]
negative_tweets = tweetwords.loc[tweets.airline_sentiment == 'negative', :]

sample400_allwords = pd.concat([positive_tweets.iloc[0:200, : ], negative_tweets.iloc[0: 200, : ]], axis = 'rows')
sample400_allwords.shape

(400, 4000)

Let's also create a version of this data limited to our list of 200 highly predictive (extreme) features.

In [26]:
sample400_ext200 = sample400_allwords.loc[ : , extremefeatures]  # why does this work?
sample400_ext200.shape

(400, 200)

Now we need a list of true labels for the model to predict:

In [28]:
true_sentiment_400 = [1] * 200 + [0] * 200
# what will that produce?

Now let's train a first model. We'll use just the 200 words that Dunnings identified as extreme.

In [30]:
sm_bayes = MultinomialNB(alpha = 0.1)   # Alpha sets the "smoothing." Notice we're not using much.
sm_bayes.fit(sample400_ext200, true_sentiment_400)   # Here we actually fit the model

sm_predictions = sm_bayes.predict(sample400_ext200)   # Now let's see what the model predicts if we give it
sum(sm_predictions == true_sentiment_400) / len(sm_predictions)  # the same data.

0.8825

Not a bad prediction. But how would this model do if we gave it new data?

Let's make a bigger dataset, with the next 1500 negative tweets and the next 1500 positive tweets.

In [32]:
next3000 =  pd.concat([positive_tweets.iloc[200: 1700, : ], negative_tweets.iloc[200: 1700, : ]], axis = 'rows')
true_sentiment_3000 = [1] * 1500 + [0] * 1500
next3000.shape

(3000, 4000)

Now a version limited to 200 words that are strongly associated with positive or negative tweets.

In [34]:
next3000_ext200 = next3000.loc[ : , extremefeatures]

In [35]:
sm_predictions = sm_bayes.predict(next3000_ext200)
sum(sm_predictions == true_sentiment_3000) / len(sm_predictions)

0.7953333333333333

Oh, that's much worse. Maybe we could improve this by increasing the number of words we use to predict? Let's train on the version of our 400-tweet dataset that has all 5000 words.

In [37]:
big_bayes = MultinomialNB(alpha = 0.1)
big_bayes.fit(sample400_allwords, true_sentiment_400)
big_predictions = big_bayes.predict(sample400_allwords)
sum(big_predictions == true_sentiment_400) / len(big_predictions)

0.9925

Ah, yes. Including lots more words makes our model almost perfectly accurate on the 400 tweets we used to train it.

Maybe that will also improve accuracy on the new data?

In [39]:
big_predictions = big_bayes.predict(next3000.loc[ : , : ])
sum(big_predictions == true_sentiment_3000) / len(big_predictions)

0.7313333333333333